In [ ]:
import json
import time

def get_last_xy_fast(wp_str):
    """
    [Fast Coordinate Extraction]
    Skips parsing the entire trajectory; directly extracts the final (x, y) 
    from the end of the string for maximum performance.
    Example Input: "... (30.97, -1.30, -0.06), (33.84, -1.48, -0.05)"
    """
    try:
        # Search for the last opening parenthesis from the right
        last_paren_idx = wp_str.rfind('(')
        if last_paren_idx == -1:
            return None, None
            
        # Extract content within the last parenthesis, e.g., "33.84, -1.48, -0.05"
        last_tuple_str = wp_str[last_paren_idx+1:].replace(')', '').strip()
        
        # Split by comma and extract the first two numerical values (x, y)
        parts = last_tuple_str.split(',')
        x_last = float(parts[0].strip())
        y_last = float(parts[1].strip())
        
        return x_last, y_last
    except Exception as e:
        return None, None

def get_intent_label(x_last, y_last, x_stop_th=5.0, y_slight_th=2.0, y_hard_th=5.0):
    """Classifies high-level driving intent (6 classes) based on the final waypoint position."""
    # Logic: If longitudinal progress is minimal, assume stopping/decelerating
    if x_last <= x_stop_th:
        return "<Decelerate_Stop>"
    
    # Classification based on lateral (y) displacement
    if y_last > y_hard_th:
        return "<Hard_Left>"
    elif y_slight_th < y_last <= y_hard_th:
        return "<Slight_Left>"
    elif -y_slight_th <= y_last <= y_slight_th:
        return "<Keep_Straight>"
    elif -y_hard_th <= y_last < -y_slight_th:
        return "<Slight_Right>"
    else: 
        return "<Hard_Right>"

def main():
    input_file = 'nuscenes_reasons_val_Qwen_32B.jsonl'           # Original dataset
    output_file = 'nuscenes_reasons_val_Qwen_32B_commands.jsonl' # New dataset with reordered fields and labels
    
    # Initialize statistics counter
    stats = {
        "<Hard_Left>": 0, "<Slight_Left>": 0, "<Keep_Straight>": 0,
        "<Slight_Right>": 0, "<Hard_Right>": 0, "<Decelerate_Stop>": 0
    }
    
    total_valid = 0
    start_time = time.time()
    
    print(f"High-speed mode enabled. Processing massive dataset...")
    
    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:
        
        for line in fin:
            if not line.strip():
                continue
                
            original_data = json.loads(line)
            wp_str = original_data.get("wp_future", "")
            
            if wp_str:
                # [Performance Gain] Directly extract the last coordinate without full string parsing
                x_last, y_last = get_last_xy_fast(wp_str)
                
                if x_last is not None and y_last is not None:
                    # 1. Calculate the command intent
                    command_intent = get_intent_label(x_last, y_last)
                    
                    # 2. Create ordered dictionary and re-order fields (token and command first)
                    ordered_data = {}
                    ordered_data["token"] = original_data.get("token", "MISSING_TOKEN")
                    ordered_data["command"] = command_intent
                    
                    # Append remaining keys
                    for key, value in original_data.items():
                        if key not in ["token", "command"]:
                            ordered_data[key] = value
                    
                    # Update statistics
                    stats[command_intent] += 1
                    total_valid += 1
                    
                    # 3. Write to the new file
                    fout.write(json.dumps(ordered_data) + '\n')
            
            # Progress reporting (every 100k records)
            if total_valid % 100000 == 0 and total_valid > 0:
                elapsed = time.time() - start_time
                speed = total_valid / elapsed
                print(f"Processed {total_valid} records... (Speed: {speed:.0f} records/sec)")

    end_time = time.time()
    total_time = end_time - start_time
    
    print("-" * 50)
    print(f"Processing complete! Total time: {total_time:.2f} seconds")
    print(f"Average speed: {total_valid / total_time:.0f} records/second")
    print(f"New dataset saved to: {output_file}")
    
    print("\n Dataset Command Distribution:")
    for intent_name, count in stats.items():
        percentage = (count / total_valid) * 100 if total_valid > 0 else 0
        print(f"  {intent_name:<18}: {count:>8} records ({percentage:>5.2f}%)")
    print("-" * 50)

if __name__ == "__main__":
    main()